# Chest X-ray Screening with Grad-CAM Explainability

**Dataset:** Kermany paediatric chest X-rays (Kaggle `paultimothymooney/chest-xray-pneumonia`)
**Task:** binary — NORMAL vs PNEUMONIA
**Pipeline:** CLAHE → augmentation → DenseNet121 / EfficientNet-B0 transfer learning → Grad-CAM → TFLite

> **Before you run anything:** `Runtime → Change runtime type → T4 GPU`.
> A full run (both backbones, download included) takes roughly 45–60 minutes.

---
### Order of cells
1. GPU check
2. Get the project code
3. Kaggle credentials
4. Download + CLAHE + patient-safe splits
5. Train DenseNet121
6. Train EfficientNet-B0
7. Compare the two
8. Grad-CAM figures
9. TFLite export
10. Download everything

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "NONE  -> Runtime > Change runtime type > T4 GPU, then rerun")

## 2. Get the project code

Pick **one** of the two cells below.

* **2a** if the project is pushed to GitHub — set `REPO_URL`.
* **2b** otherwise — zip the `HM Project` folder on your laptop and upload it here.

In [ ]:
# --- 2a. Clone from GitHub -------------------------------------------------
REPO_URL = ""   # e.g. "https://github.com/<you>/chest-xray-gradcam.git"

import os, shutil
if REPO_URL:
    shutil.rmtree("/content/project", ignore_errors=True)
    !git clone -q $REPO_URL /content/project
    os.chdir("/content/project")
    print("cloned into", os.getcwd())
    !ls
else:
    print("REPO_URL is empty - use cell 2b instead.")

In [ ]:
# --- 2b. Upload a zip of the project folder --------------------------------
# On Windows: right-click the "HM Project" folder > Send to > Compressed (zipped) folder
import os, glob, zipfile, shutil
from google.colab import files

up = files.upload()                      # pick the .zip
name = next(iter(up))
shutil.rmtree("/content/project", ignore_errors=True)
os.makedirs("/content/project", exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall("/content/_unzip")

# the zip usually contains one top-level folder - step into it
root = next(iter(glob.glob("/content/_unzip/**/src/train.py", recursive=True)), None)
assert root, "src/train.py not found inside the zip"
src_root = os.path.dirname(os.path.dirname(root))
for item in os.listdir(src_root):
    shutil.move(os.path.join(src_root, item), "/content/project")
os.chdir("/content/project")
print("project at", os.getcwd())
!ls

## 3. Kaggle credentials

Kaggle → your avatar → **Settings** → **API** → *Create New Token* downloads `kaggle.json`.

Preferred: add it once to Colab **Secrets** (🔑 in the left sidebar) as two secrets,
`KAGGLE_USERNAME` and `KAGGLE_KEY`, and enable notebook access. Otherwise the cell
falls back to uploading the file.

In [ ]:
import os, json, pathlib

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("using Colab secrets for", os.environ["KAGGLE_USERNAME"])
except Exception as e:
    print("secrets unavailable (%s) - upload kaggle.json instead" % type(e).__name__)
    from google.colab import files
    up = files.upload()
    creds = json.loads(next(iter(up.values())))
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]

d = pathlib.Path.home() / ".kaggle"; d.mkdir(exist_ok=True)
(d / "kaggle.json").write_text(json.dumps({
    "username": os.environ["KAGGLE_USERNAME"], "key": os.environ["KAGGLE_KEY"]}))
(d / "kaggle.json").chmod(0o600)
print("kaggle.json ready")

## 4. Download, CLAHE, and split

This runs `scripts/prepare_data.py --download`, which:

* pulls the 1.2 GB archive from Kaggle and unzips it,
* applies **CLAHE** to every image and writes lossless 224×224 PNGs,
* merges the useless 16-image official `val` folder back into train and carves a
  proper 10% validation set with **StratifiedGroupKFold** — class-balanced *and*
  patient-disjoint, so no child appears on both sides of the split,
* leaves the official `test` folder completely untouched.

Takes about 5–8 minutes, mostly download.

In [ ]:
!pip install -q kaggle
!python scripts/prepare_data.py --download

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv("reports/dataset_summary.csv"))
display(Image("reports/clahe_examples.png", width=760))

## 5. Train DenseNet121

Two phases:

| Phase | Backbone | LR | Epochs |
|---|---|---|---|
| warm-up | frozen | 1e-3 | 5 |
| fine-tune | deepest 50% unfrozen, BatchNorm frozen | 1e-4 | up to 20 (early-stopped on val AUROC) |

Class weights compensate the ~3:1 pneumonia imbalance. Expect ~12–18 min on a T4.

In [ ]:
!python src/train.py --backbone densenet121

## 6. Train EfficientNet-B0

In [ ]:
!python src/train.py --backbone efficientnetb0

## 7. Compare the two backbones

Everything below reads the metrics the two runs wrote to `reports/<run>/metrics.json`.
Pick the winner on **validation** AUROC if they are close — the test set is only
being reported, never selected on.

In [ ]:
import json, pandas as pd
from pathlib import Path

rows = []
for f in sorted(Path("reports").glob("*/metrics.json")):
    m = json.loads(f.read_text())
    lo, hi = m["test_auroc_95ci"]
    rows.append({
        "run": f.parent.name,
        "val AUROC": round(m["val"]["auroc"], 4),
        "test AUROC": round(m["test"]["auroc"], 4),
        "95% CI": "%.3f-%.3f" % (lo, hi),
        "accuracy": round(m["test"]["accuracy"], 4),
        "sensitivity": round(m["test"]["sensitivity_recall"], 4),
        "specificity": round(m["test"]["specificity"], 4),
        "F1": round(m["test"]["f1"], 4),
        "threshold": round(m["test"]["threshold"], 3),
        "min": m.get("train_minutes"),
    })
comparison = pd.DataFrame(rows).sort_values("val AUROC", ascending=False)
comparison.to_csv("reports/backbone_comparison.csv", index=False)
comparison

In [ ]:
from IPython.display import Image, display
BEST = comparison.iloc[0]["run"]          # change by hand if you prefer the other
print("best run:", BEST)
for f in ("training_curves.png", "test_curves.png", "test_confusion.png"):
    display(Image(f"reports/{BEST}/{f}", width=820))

## 8. Grad-CAM explanations

Grad-CAM differentiates the **logit** (not the saturated sigmoid) with respect to
the last convolutional feature map, weights each channel by its mean gradient and
keeps the positive part. The `gradcam_errors.png` panel — what the model looked at
when it was *wrong* — is usually the most interesting slide in the report.

In [ ]:
# BEST comes from the comparison cell above; run_name == backbone name here.
!python scripts/make_gradcam_figures.py --backbone {BEST}

In [ ]:
from IPython.display import Image, display
for f in ("gradcam_pneumonia.png", "gradcam_normal.png",
          "gradcam_errors.png", "gradcam_vs_pp.png"):
    p = f"reports/{BEST}/{f}"
    try:
        display(Image(p, width=880))
    except Exception:
        print("missing:", p)

## 9. Export to TensorFlow Lite

Writes a float32 `.tflite` and a dynamic-range int8 one, and refuses to ship the
float model if its predictions drift from Keras by more than 1e-3.

In [ ]:
!python src/export_tflite.py --backbone densenet121
!python src/export_tflite.py --backbone efficientnetb0

## 10. Download the results

In [ ]:
!zip -qr /content/results.zip reports checkpoints -x "*.npz"
from google.colab import files
files.download("/content/results.zip")

In [ ]:
# Optional: keep the checkpoints on Google Drive instead of re-downloading them
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/HM Project" && cp -r reports checkpoints "/content/drive/MyDrive/HM Project/"